In [ ]:
# 导入相关库，全局变量（API_KEY、INDEX_NAME、VIDEO_ID等）、定义Context类、辅助函数（create_index、parse_json_from_llm、ts_to_seconds等）
# .venv/scripts/python.exe, Python 3.12.4
from dataclasses import dataclass
import json
from typing import Any, Dict, Optional
from jinja2 import Template
from twelvelabs import ResponseFormat
from twelvelabs import IndexesCreateResponse, TwelveLabs, core
from twelvelabs.indexes import IndexesCreateRequestModelsItem

API_KEY = "tlk_36GR70D23DGHGW23WQ2TK3PFRW5F"
INDEX_NAME = "memory_assistant_index"
INDEX_ID = "6902e65d8ef1cd6b38b81eb5"
VIDEO_ID = "694f9289db0246c06ce26865"
VIDEO_GLOB = "videos/*.mp4"

def create_index(
        client, 
        index_name, 
        model_name="Pegasus1.2", 
        model_options=["visual", "audio"]
        )-> IndexesCreateResponse:
    try:
        index = client.indexes.create(
            index_name=index_name,
            models=[
                IndexesCreateRequestModelsItem(
                    model_name=model_name,
                    model_options=model_options,
                ),
            ],
        )
    except Exception as e:
        print(f"Error creating index: {e}")
        raise

    print(f"Created index: id={index.id} name={index.name}")
    return index


@dataclass
class ActivityContext:
    activity: str
    people: str
    time: str
    location: str
    sub_event: Optional[str]

@dataclass
class EntityContext:
    activity: str
    people: str
    time: str
    start_time: str
    end_time: str
    sub_event: Optional[str]


def parse_json_from_llm(data_str: str) -> dict:
    if data_str is None:
        raise ValueError("LLM returned None")

    s = data_str.strip()
    if s.startswith("```json"):
        s = s[7:].strip()
    if s.startswith("```"):
        s = s[3:].strip()
    if s.endswith("```"):
        s = s[:-3].strip()

    return json.loads(s)
def ts_to_seconds(ts: str) -> float:
    ts = ts.strip()
    parts = ts.split(":")
    if len(parts) == 2:
        mm, ss = parts
        return int(mm) * 60 + float(ss)
    elif len(parts) == 3:
        hh, mm, ss = parts
        return int(hh) * 3600 + int(mm) * 60 + float(ss)
    else:
        raise ValueError(f"Unsupported timestamp format: {ts}")

In [ ]:
# 弃用：以函数定义事件划分和实体提取prompt
def build_event_split_prompt(
    ctx: ActivityContext,
    segmentation_rules: str,
    cue_rules: str,
) -> str:
    """
    组装 LangGPT 风格 Prompt：Task / Context / Rules / Workflow / Output Format / Example
    """
    task = (
        "对[输入]的第一人称视频进行子事件切分，并对划分出的每个子事件进行描述概括和关键帧线索提取。\n"
        "子事件切分任务定义:基于整体活动主题，将视频中的活动划分为能够构成完整活动流程的关键事件序列。将人脑中把连续的外界信息切分成若于个有意义日互相关联的事件的过程称作事件切分（eventseqmentation），连续信息被切分成不同事件的时间点就是事件边界（event boundary），它标志着前一事件的结束和后一事件的开始。\n"
    )

    context = (
        f"视频中的活动主题是[{ctx.activity}]，人物是[{ctx.people}]，时间是[{ctx.time}]，地点是[{ctx.location}]。\n"
    )

        #     "### 划分要求\n"
        # "    1. 划分的事件必须能够做为构成视频拍摄者完整活动流程的关键行为步骤。\n"
        # "    2. 若基于行为主题切换来分割事件，则划分出的行为主题需具备行为目标和认知负荷，拍摄者注意力需集中在行为本身，即不能包括用户无意识的行为。正面示例包括“拿外卖”、“点餐”、“进入商场”等；反面示例包括“摆弄头发”。\n"
        # "    3. 若基于行为主题切换来分割事件，则识别行为边界时，需保障切割后的两项行为的主题或目标不同。\n"
        # "    4. 若基于活动场景切换来分割事件，则识别因镜头抖动、用户身体部位遮挡、以及手机遮挡而导致场景的意外变化，并剔除这一影响。\n"
        # "    5. 若基于活动场景切换来分割事件，在识别场景边界时，需保障切割后的两个场景之间可回忆内容（如场景中的关键物品、用户在场景中的关键行为、或用户出现在该场景的目的等）存在切换。\n"
        # "    6. “看手机”不可以被识别为事件。\n\n"
    
    rules = (
        "## 事件边界划分标准\n"
        f"{segmentation_rules}\n\n"
        "### 事件描述标准\n"
        "描述内容需包括视频拍摄者在子事件中的所处场景和关键行为，且该子事件需与活动主题高度相关、并具备一定目标和认知负荷\n\n"
        "## 线索提取标准\n"
        f"{cue_rules}\n"
    )

    workflow = (
        "1. 基于【划分标准】与【划分要求】识别事件边界并划分事件。\n"
        "2. 基于【事件描述标准】，对划分出的事件进行概括描述。\n"
        "3. 合并非关键事件：判断事件划分粒度是否合理，如果事件并非构成活动流程的关键行为，则考虑将事件进行合并。例如“通过地铁站闸机”并不能算作关键事件，可以合并到“离开地铁站”这一更大的事件单位中\n"
        "4. 对划分出的每个事件，基于【线索提取标准】构建线索：选取关键帧图像、提取关键帧附近的音频数据、生成语义提示。\n"
        "5. 检查线索内容是否直接透露检索目标，如果是，则重新构建线索。\n"
        "6. 输出子事件信息。"
    )

    output_format = "以标准JSON格式输出活动整体概括描述（activity_description）、事件数量(event_count)、以及事件序列（events）。事件序列中，每个事件的属性包括事件的描述（description）、开始时间（start_time）、结束时间（end_time）、事件关键帧线索（scene_clues）（包括关键帧时间戳和关键帧语义描述）。\n"

    example_json = {
        "video_length": "00:19:15",
        "activity_description": "去快餐店购买午餐",
        "event_count": 4,
        "events": [
            {
                "id": 1,
                "description": "在餐厅点餐",
                "start_time": "00:00:00",
                "end_time": "00:03:48",
                "scene_clues": [
                    {"frame": "00:01:45", "description": "服务员递来菜单"},
                    {"frame": "00:01:55", "description": "查看菜单"}
                ],
            },
            {
                "id": 2,
                "description": "等待取餐",
                "start_time": "00:03:48",
                "end_time": "00:05:32",
                "scene_clues": [
                    {"frame": "00:04:37", "description": "站在吧台等待"},
                ],
            },
            {
                "id": 3,
                "description": "取餐",
                "start_time": "00:05:32",
                "end_time": "00:06:57",
                "scene_clues": [
                    {"frame": "00:05:33", "description": "服务员呼叫"},
                    {"frame": "00:06:55", "description": "接过食品袋"},
                ],
            },
            {
                "id": 4,
                "description": "离开餐厅",
                "start_time": "00:06:57",
                "end_time": "00:07:12",
                "scene_clues": [
                    {"frame": "00:07:11", "description": "走出餐厅大门"},
                ],
            }
        ],
    }

    prompt = f"""
# 1. Task（任务）
{task}

# 2. Context（活动上下文）
{context}

# 3. Rules（规则）
{rules}

# 4. Workflow（工作流）
{workflow}

# 5. Format（输出格式）
{output_format}


# 6. Example（输出示例）
```json
{json.dumps(example_json, ensure_ascii=False, indent=2)}
"""
        
    return prompt

# - 活动场景切换：视频拍摄者的活动场景发生变化时分割事件，如室内-室外场景的切换、不同房间的切换、活动区域的整体切换等。
#     - 基于第一人称视频的补充标准
#         1. 识别并过滤身体部位遮挡造成对场景切换的误判
#         2. 识别并过滤手机遮挡造成对场景切换的误判
# - 行为主题切换：视频拍摄者的行为主题或目标发生变化时分割事件，如静止-行走状态的切换、聊天-沉默状态的切换、停止用餐等。
#     - 基于第一人称视频的补充标准
#         1. 识别并过滤用户的无意识行为，如拨弄头发等
#         2. 识别并过滤看手机事件

SEGMENTATION_RULES = """\
基于视频拍摄者的活动场景发生变化时分割事件，如室内-室外场景的切换、不同房间的切换、活动区域的整体切换等。或基于视频拍摄者的行为主题或目标发生变化时分割事件，如静止-行走状态的切换、聊天-沉默状态的切换、停止用餐等。\
"""

# - 视觉线索（关键帧）：
#     1. 画面清晰，画面中无人物身体部位、手机等与事件无关的物体遮挡。
#     2. 画面中不包含该场景中的后续实体对象材料。
# - 语义线索（文本）：
#     场景特点提示（色彩丰富度、感官联想），例如：
#     1. 视觉特征：“在超市里，你有没有看到什么颜色特别显眼的区域？”
#     2. 温度特征：“你有没有在某个温度比较低的区域停留？”

CUE_RULES = """\
关键帧提取要求：画质清晰，画面中无人物身体部位、手机等与事件无关的物体遮挡；画面中不包含该场景中的后续实体对象材料。语义线索生成要求：从场景特点（色彩丰富度、感官联想）入手进行提示，例如视觉特征（“超市里你有没有看到什么颜色特别鲜艳的区域”）、温度特征（“你有没有在某个温度比较低的区域停留”）。线索不可以直接透露检索目标。\
"""


def build_entity_extract_prompt(
    ctx: EntityContext,
    entity_rules: str,
    entity_cue_rules: str,
) -> str:
    """
    组装 LangGPT 风格 Prompt：Task / Context / Rules / Workflow / Output Format / Example
    """
    task = (
        "对[输入]的第一人称视频，从视频中提取出能够被视频拍摄者注意到并进行编码的关键实体项目，作为后续引导拍摄者进行回忆的记忆材料；围绕提取的每项记忆材料，选取关键帧作为视觉线索，并生成文本提示作为语义线索。\n"
    )

    context = (
        f"视频中的活动主题是[{ctx.activity}]，人物是[{ctx.people}]，时间是[{ctx.time}]，当前活动中的事件是[{ctx.sub_event.strip("。") if ctx.sub_event else ctx.sub_event}]，事件所在视频时间段是[{ctx.start_time} - {ctx.end_time}]。\n"
    )

    rules = (
        "【实体项目筛选与过滤标准】\n"
        f"{entity_rules}\n\n"
        "【实体层线索提取标准】\n"
        f"{entity_cue_rules}\n"
    )

    workflow = (
        "1. 对[输入]的第一人称视频，识别其中的人物、物体两类实体项目。\n"
        "2. 对识别出的每一个项目，基于[实体项目筛选与过滤标准]判断是否筛选该项目作为记忆材料，并给出决策理由。\n"
        "3. 对于筛选出的每一个项目，请判断：假设你是视频的拍摄者，在活动中你是否会注意到该实体项目、以及在活动结束至少间隔2小时后再次回忆活动时你是否会对该实体项目有印象，是则保留该项目，否则过滤掉该项目不进行保留。\n"
        "4. 对于筛选出的每一个项目，基于[实体层线索提取标准]构建线索:选取关键帧图像、提取关键帧附近的音频数据、生成语义提示。\n"
        "5. 输出实体项目信息。\n"
    )

    output_format = (
        "以标准JSON格式输出关键人物和物体项目序列（[key_persons]、[key_objects]），其中每个项目的属性包括：项目名（item_name），项目的语义定义；关键帧（key_frame），项目在视频中出现的关键帧，以时间戳形式提供；坐标（coordinates），项目在关键帧中的坐标定位；细节（details），项目关键细节，包括语义细节（visual）和视觉细节（semantic）；交互（interaction），项目与拍摄者之间存在的交互关系或行为描述。"
    )

    example_json = {
        "key_persons": [
            {
                "id": "p1",
                "description": "朋友张伟",
                "key_frame": "00:02:15",
                "coordinates": [
                    {"x": 350, "y": 140, "width": 130, "height": 200}
                ],
                "details":{
                      "visual": ["灰色毛衣", "挥手动作", "笑容"],
                      "semantic": ["老朋友", "同事"]
                },
                "interaction": "热情打招呼，拥抱问候",
            },
            {
                "id": "p2",
                "description": "朋友小美",
                "key_frame": "00:02:45",
                "coordinates": [
                    {"x": 220, "y": 130, "width": 120, "height": 195}
                ],
                "details":{
                      "visual": ["蓝色衬衫", "眼镜"],
                      "semantic": ["大学同学", "好友"]
                },
                "interaction": "握手致意，互相介绍",
            }
        ],
        "key_objects": [
            {
            "id": "o1",
            "item_name": "鸳鸯火锅",
            "key_frame": "00:14:21",
            "coordinates": {"x": 311, "y": 273, "width": 200, "height": 150}, 
            "details": {
                "visual": ["鸳鸯锅", "红汤白汤", "蒸汽缭绕"],
                "semantic": []
            },
            "interaction": "涮煮食材"
            },
            {
            "id": "o2",
            "item_name": "肥牛片拼盘",
            "key_frame": "00:16:12",
            "coordinates": {"x": 248, "y": 14, "width": 200, "height": 150}, 
            "details": {
                "visual": [],
                "semantic": ["两盘"]
            },
            "interaction": "选取食材，放入锅中"
            }
        ]
    }

    prompt = f"""
# 1. Task（任务）
{task}

# 2. Context（活动上下文）
{context}

# 3. Rules（规则）
{rules}

# 4. Workflow（工作流）
{workflow}

# 5. Format（输出格式）
```json
{json.dumps(output_format, ensure_ascii=False, indent=2)}


# 6. Example（输出示例）
```json
{json.dumps(example_json, ensure_ascii=False, indent=2)}
"""
        
    return prompt

# - 正面示例
#     1. 对话交互：在“餐厅用餐”时，服务员向用户推荐菜品。
#     2. 协同活动：在“公园散步”时，女儿陪用户走路聊天。
#     3. 服务交互：在“理发”时，理发师为用户理发并交流。
#     4. 情感交互：在“家庭聚会”时，用户与家人互相关心近况。
# - 过滤标准
#     1. 对话内容与活动主题无关，仅包括简短问候等的人物。
#     2. 在画面中出现，但无任何交互、拍摄者明显未关注的背景人物。
# - 反面示例
#     1. 进入门店时，服务员仅对用户问好，无后续对话。
#     2. 乘坐地铁时，车厢内的其他乘客。
#     3. 超市购物时，便利店员工快速扫码，仅进行一两句交易对话。
# - 正面示例
#     1. 购买决策：在书店时，用户翻阅一本书的目录和序言，考虑是否购买。
#     2. 关注焦点：在参观博物馆时，用户在一幅画前驻足观看并拍照。
#     3. 情感关联：在商场购物时，用户看到价格标签时说“太贵了”。
# - 过滤标准
#     1. 纯装饰性背景物体，且用户未关注。
#     2. 在画面中一闪而过的物体。
#     3. 功能性基础设施（如道路、建筑墙面等），除非用户特别关注。
# - 反面示例
#     1. 墙上的装饰画，用户从未看该物体。
#     2. 街道行走时，快速开过去的汽车。
#     3. 道路、墙面、天花板、电线杆等功能性基础设施。但特殊情况需提取项目，如用户因迷路而特别关注的路标、因美观而拍摄的建筑外墙等。
ENTITY_RULES = """\
1. 人物类项目筛选标准：和视频拍摄者（用户）之间须至少存在以下交互关系之一：对话交互：人物与拍摄者进行有意义的语言交流；且对话内容涉及具体主题（非仅限于"你好"等礼貌用语）；协同活动：人物与拍摄者共同参与活动主题，且持续时间≥1分钟；服务交互：该服务围绕活动主题展开，人物为拍摄者提供服务；情感交互：人物与拍摄者之间存在明显的情感表达，包括：问候拥抱、表达关心、分享情感等。
2. 物体类项目筛选标准：和视频拍摄者（用户）之间须至少存在以下交互关系之一：购买决策：用户考虑购买、比较或最终购买的物体。包括拿起查看、询问 价格、对比选择等行为；关注焦点：物体成为用户明显的注意焦点，用户对物体进行观察、拍摄或讨论；情感关联：用户对物体表现出明显的情感反应，包括惊讶、兴奋等。通常伴随语言表达或肢体动作。
"""

ENTITY_CUE_RULES = """\
1. 视觉线索（关键帧）：画面清晰，画面中无人物身体部位、手机等与事件无关的物体遮挡；画面中呈现实体项目材料（用于间接视觉线索生成）。
2。 语义线索（文本）：实体语义类别标签（食物/工具/家人/朋友等）；实体的个性化提示：考虑是否能结合用户画像生成个性化语义提示，例如是否为用户常吃的食物、是否是昨天活动中才见过的人物等。\
"""

In [ ]:
client = TwelveLabs(api_key=API_KEY)

In [ ]:
# 如果索引不存在创建新索引（INDEX_ID和VIDEO_ID）

if INDEX_ID is None:
    index = create_index(client, INDEX_NAME)
    INDEX_ID = index.id
    print(f"Created index: id={index.id}")

if INDEX_ID:
    if VIDEO_ID is None:
        video_path = r"D:\VSCode\VSProj\LifelogProj\Videos\short_version2min.mp4"
        with open(video_path, "rb") as f:
            task = client.tasks.create(
                index_id=INDEX_ID, 
                video_file=("short_version2min.mp4", f, "video/mp4"),
            )
            print(f"Created task: id={task.id}, video_id={task.video_id}")
            VIDEO_ID = task.video_id


Generated Prompt:
# 1. Task（任务）
你将处理一个第一人称拍摄的视频任务。目标是：
1. 根据视频内容识别出活动中的子事件；
2. 每个事件需进行简洁描述；
3. 为每个事件提取关键帧线索（包括视觉与语义提示）。

## 子事件切分定义
- 事件切分（Event Segmentation）：指人脑将连续信息划分为若干有意义的片段的过程。
- 事件边界（Event Boundary）：表示一个子事件的结束和下一个事件的开始。
- 子事件应具备明确的行为目标、认知负荷，与整体活动密切相关。

# 2. Context（活动上下文）
视频中的活动主题是[去超市购物]，人物是[我和朋友]，时间是[2024年6月15日下午5点]，地点是[胖东来]。

# 3. Rules（规则）

## 事件边界划分标准
- 基于“活动场景切换”或“行为主题切换”判断事件边界；
- 活动场景切换指视频拍摄者的活动场景发生变化时分割事件，如室内-室外场景的切换、不同房间的切换、活动区域的整体切换等；
- 行为主题切换指视频拍摄者的行为主题或目标发生变化时分割事件，如静止-行走状态的切换、聊天-沉默状态的切换、停止用餐等；
- 排除无意识行为（如拨弄头发）、身体遮挡或手机遮挡造成的误判；
- 不可将“看手机”视为有效事件；
- 如果事件并非构成活动流程的关键行为，则考虑将事件进行合并。例如“通过地铁站闸机”并不能算作关键事件，可以合并到“离开地铁站”这一更大的事件单位中。

### 事件描述标准
描述内容需包括视频拍摄者在子事件中的所处场景和关键行为，且该子事件需与活动主题高度相关、并具备一定目标和认知负荷。

## 线索提取标准
- **视觉线索（关键帧）：**关键帧需画质清晰，无干扰物体（如身体部位、手机等）遮挡；
- **语义线索（文本）：**从场景感官特征出发构建问题提示（色彩丰富度、感官联想），例如：
    - 视觉特征：“在超市里，你有没有看到什么颜色特别显眼的区域？”
    - 温度特征：“你有没有在某个温度比较低的区域停留？”
- 可使用音频中出现的环境声音（如对话）辅助识别事件特征；
  - 示例：
    - 听到服务员说“请问要点什么” → 表示点餐开始；
    - 用户说“吃完了” → 表示用餐结束；
- 线索语义描述应避免直接陈述动作本身

In [ ]:
# 定义活动上下文，调用API划分事件
ctx = ActivityContext(
    activity="去超市购物",
    people="我和朋友",
    time="2024年6月15日下午5点",
    location="胖东来",
    sub_event=None
)

with open("prompts/event_split_prompt.md", "r", encoding="utf-8") as f:
    template = Template(f.read())

prompt = template.render(
    activity=ctx.activity,
    people=ctx.people,
    time=ctx.time,
    location=ctx.location,
)
print("------Event split prompt------\n")
print(prompt)

if INDEX_ID!= None and VIDEO_ID != None:
    event_resp = client.analyze(
        video_id=VIDEO_ID, 
        prompt=prompt,
        temperature=0.2,
        response_format=ResponseFormat(
            json_schema={
                "activity_description": "string",
                "event_count": "integer",
                "events": [
                    {
                        "id": "string",
                        "description": "string",
                        "start_time": "HH:MM:SS",
                        "end_time": "HH:MM:SS",
                        "scene_clues": [
                            {"frame": "HH:MM:SS", "description": "string"}
                        ],
                    }
                ],
            },
        )
    )

    if event_resp.data:
        event_parsed = parse_json_from_llm(event_resp.data)
        print("-----Event split response(parsed JSON)------\n")
        print(json.dumps(event_parsed, ensure_ascii=False, indent=2))

Parsed JSON:
{
  "activity_description": "去胖东来超市购物",
  "event_count": 7,
  "events": [
    {
      "id": "1",
      "description": "进入超市",
      "start_time": "00:00",
      "end_time": "00:05",
      "scene_clues": [
        {
          "frame": "00:00",
          "description": "推着购物车进入超市"
        },
        {
          "frame": "00:01",
          "description": "经过收银台"
        }
      ]
    },
    {
      "id": "2",
      "description": "选择蔬果",
      "start_time": "00:05",
      "end_time": "00:12",
      "scene_clues": [
        {
          "frame": "00:08",
          "description": "选择柠檬"
        },
        {
          "frame": "00:12",
          "description": "选择莴笋"
        }
      ]
    },
    {
      "id": "3",
      "description": "选择熟食",
      "start_time": "00:21",
      "end_time": "00:37",
      "scene_clues": [
        {
          "frame": "00:21",
          "description": "选择熟食"
        },
        {
          "frame": "00:37",
          "description": "将熟食放入购物车"
       

In [ ]:
# 以一个事件为例，构建实体上下文、实体提取prompt
events = event_parsed.get("events", [])
event = events[5]
start_ts = event.get("start_time")
end_ts = event.get("end_time")

ctx = EntityContext(
        activity="去超市购物",
        people="我和朋友",
        time="2024年6月15日17:00",
        start_time=start_ts,
        end_time=end_ts,
        sub_event=event.get("description")
    )

with open("prompts/entity_extract_prompt.md", "r", encoding="utf-8") as f:
        template = Template(f.read())

entity_prompt = template.render(
    activity=ctx.activity,
    people=ctx.people,
    time=ctx.time,
    sub_event=ctx.sub_event.strip("。") if ctx.sub_event else ctx.sub_event,
    event_time_range=f"{ctx.start_time} - {ctx.end_time}",
)

print("------Entity Extraction Prompt------\n")
print(entity_prompt)

Generated Entity Extraction Prompt:
# 1. Task（任务）
你需要从第一人称拍摄的视频片段中，识别出用户在活动中注意到的**关键人物与关键物体**，这些实体将作为未来“回忆重建”的语义材料。

每个实体项目需满足以下要求：

- 与用户存在显著的交互或注意力行为；
- 足够显著以至于可能被编码入用户记忆；
- 可构建包含视觉与语义维度的回忆线索（包括图像、语义提示、标签、背景声音等）。

# 2. Context（活动上下文）
视频中的活动主题是[去超市购物]，人物是[我和朋友]，时间是[2024年6月15日17:00]，当前关注的子事件是[选择饮料]，该事件在视频中的起止时间为「01:15 - 01:27」。
请仅分析此时间段内的内容，用于识别与该子事件相关的关键人物与关键物体。

# 3. Rules（规则）

## 实体项目筛选与过滤标准
### 1. 人物类（key_persons）
和视频拍摄者（用户）之间须至少存在以下交互关系之一：
- **对话交互**：人物与拍摄者进行有意义的语言交流；且对话内容涉及具体主题（非仅限于"你好"等礼貌用语）；
- **协同活动**：人物与拍摄者共同参与活动主题，且持续时间≥1分钟；
- **服务交互**：该服务围绕活动主题展开，人物为拍摄者提供服务；
- **情感交互**：人物与拍摄者之间存在明显的情感表达，包括：问候拥抱、表达关心、分享情感等。

正面示例：
1. 对话交互：在“餐厅用餐”时，服务员向用户推荐菜品。
2. 协同活动：在“公园散步”时，女儿陪用户走路聊天。
3. 服务交互：在“理发”时，理发师为用户理发并交流。
4. 情感交互：在“家庭聚会”时，用户与家人互相关心近况。

过滤标准：
1. 对话内容与活动主题无关，仅包括简短问候等的人物。
2. 在画面中出现，但无任何交互、拍摄者明显未关注的背景人物。

反面示例：
1. 进入门店时，服务员仅对用户问好，无后续对话。
2. 乘坐地铁时，车厢内的其他乘客。
3. 超市购物时，便利店员工快速扫码，仅进行一两句交易对话。
   
### 2. 物体类（key_objects）
和视频拍摄者（用户）之间须至少存在以下交互关系之一：
- **购买决策**：用户考虑购买、比较或最终购买的物体。包括拿起查看、询问 价格、对比选择等

In [ ]:
# 调用API提取实体
if INDEX_ID!= None and VIDEO_ID != None:
    entity_resp = client.analyze(
        video_id=VIDEO_ID,
        prompt=entity_prompt,
        response_format=ResponseFormat(
            json_schema={
                "key_persons": [
                    {
                        "id": "string",
                        "item_name": "string",
                        "key_frame": "HH:MM:SS",
                        "coordinates": {
                            "x": "integer",
                            "y": "integer",
                            "width": "integer",
                            "height": "integer"
                        },
                        "details": {
                            "visual": ["string"],
                            "semantic": ["string"],
                        },
                        "interaction": "string"
                    }
                ],
                "key_objects": [
                    {
                        "id": "string",
                        "item_name": "string",
                        "key_frame": "HH:MM:SS",
                        "coordinates": {
                            "x": "integer",
                            "y": "integer",
                            "width": "integer",
                            "height": "integer"
                        },
                        "details": {
                            "visual": ["string"],
                            "semantic": ["string"]
                        },
                        "interaction": "string"
                    }
                ]
            }
        )
    )

    print(f"------Entity extract response------\n")
    print(f"Event {event.get('description')}, start: {start_ts}, end: {end_ts}\n")
    if entity_resp.data:
        entity_parsed = parse_json_from_llm(entity_resp.data)
        print(json.dumps(entity_parsed, ensure_ascii=False, indent=2))
    else:
        print("No data returned from LLM.")

Event 选择饮料, start: 01:15, end: 01:27:

{
  "key_persons": [],
  "key_objects": [
    {
      "id": "o1",
      "item_name": "NF C果汁",
      "key_frame": "01:17",
      "coordinates": {
        "x": 311,
        "y": 273,
        "width": 200,
        "height": 150
      },
      "details": {
        "visual": [
          "黄色果汁",
          "透明瓶"
        ],
        "semantic": [
          "饮料"
        ]
      },
      "interaction": "购买决策"
    },
    {
      "id": "o2",
      "item_name": "三元牛奶",
      "key_frame": "01:24",
      "coordinates": {
        "x": 248,
        "y": 14,
        "width": 200,
        "height": 150
      },
      "details": {
        "visual": [
          "白色牛奶",
          "纸盒"
        ],
        "semantic": [
          "饮料"
        ]
      },
      "interaction": "购买决策"
    }
  ]
}


In [ ]:
event_count = parsed.get("event_count")
for event in parsed.get("events", []):
    start_ts = event.get("start_time")
    end_ts = event.get("end_time")

    ctx = EntityContext(
            activity="去超市购物",
            people="我和朋友",
            time="2024年6月15日17:00",
            start_time=start_ts,
            end_time=end_ts,
            sub_event=event.get("description")
        )
    
    
    with open("prompts/event_split_prompt.md", "r", encoding="utf-8") as f:
        template = Template(f.read())

    entity_prompt = template.render(
        activity=ctx.activity,
        people=ctx.people,
        time=ctx.time,
        sub_event=ctx.sub_event.strip("。") if ctx.sub_event else ctx.sub_event,
        event_time_range=f"{ctx.start_time} - {ctx.end_time}",
    )

    resp = client.analyze(
        video_id=VIDEO_ID,
        prompt=entity_prompt,
        response_format=ResponseFormat(
            json_schema={
                "key_persons": [
                    {
                        "item_name": "string",
                        "key_frame": "HH:MM:SS",
                        "coordinates": {
                            "x": "integer",
                            "y": "integer",
                            "width": "integer",
                            "height": "integer"
                        },
                        "details": {
                            "visual": ["string"],
                            "semantic": ["string"],
                        },
                        "interaction": "string"
                    }
                ],
                "key_objects": [
                    {
                        "item_name": "string",
                        "key_frame": "HH:MM:SS",
                        "coordinates": {
                            "x": "integer",
                            "y": "integer",
                            "width": "integer",
                            "height": "integer"
                        },
                        "details": {
                            "visual": ["string"],
                            "semantic": ["string"]
                        },
                        "interaction": "string"
                    }
                ]
            }
        )
    )

    print(f"Event {event.get('description')}, start: {start_ts}, end: {end_ts}:\n")
    if resp.data:
        print(json.dumps(parse_json_from_llm(resp.data), ensure_ascii=False, indent=2))
    else:
        print("No data returned from LLM.")

Generated Prompt:

# Task
对[输入]的第一人称视频，从视频中提取出能够被视频拍摄者注意到并进行编码的关键实体项目，作为后续引导拍摄者进行回忆的记忆材料；围绕提取的每项记忆材料，选取关键帧作为视觉线索，并生成文本提示作为语义线索。


# Context
视频中的活动主题是去超市购物，人物是我和朋友，时间是2024年6月15日17:00，当前活动中的事件是[进入超市，推着购物车走向水果区]。


# Rules
【实体项目筛选与过滤标准】
- 项目类型分为人物和物体两种：
    - 人物实体项目：
        - 筛选标准：和视频拍摄者（用户）之间须至少存在以下交互关系之一：
            1. 对话交互：人物与拍摄者进行有意义的语言交流；且对话内容涉及具体主题（非仅限于"你好"等礼貌用语）。
            2. 协同活动：人物与拍摄者共同参与活动主题，且持续时间≥1分钟。
            3. 服务交互：该服务围绕活动主题展开，人物为拍摄者提供服务。
            4. 情感交互：人物与拍摄者之间存在明显的情感表达，包括：问候拥抱、表达关心、分享情感等。
    - 正面示例
        1. 对话交互：在“餐厅用餐”时，服务员向用户推荐菜品。
        2. 协同活动：在“公园散步”时，女儿陪用户走路聊天。
        3. 服务交互：在“理发”时，理发师为用户理发并交流。
        4. 情感交互：在“家庭聚会”时，用户与家人互相关心近况。
    - 过滤标准
        1. 对话内容与活动主题无关，仅包括简短问候等的人物。
        2. 在画面中出现，但无任何交互、拍摄者明显未关注的背景人物。
    - 反面示例
        1. 进入门店时，服务员仅对用户问好，无后续对话。
        2. 乘坐地铁时，车厢内的其他乘客。
        3. 超市购物时，便利店员工快速扫码，仅进行一两句交易对话。
- 物体实体项目：
    - 筛选标准：和视频拍摄者（用户）之间须至少存在以下交互关系之一：
        1. 购买决策：用户考虑购买、比较或最终购买的物体。包括拿起查看、询问 价格、对比选择等行为。
        2. 关注焦点：物体成为

In [14]:
from dashscope import get_tokenizer
from dataclasses import dataclass
from jinja2 import Template

@dataclass
class EntityContext:
    activity: str
    people: str
    time: str
    location: str
    video_length: str

# 获取tokenizer对象，目前只支持通义千问系列模型
tokenizer = get_tokenizer('qwen-turbo')
ctx = EntityContext(
    activity="逛超市",
    people="我",
    time="2024年6月15日17:00",
    location="胖东来",
    video_length="00:06:15",
)
with open("prompts/full_context_prompt.md", "r", encoding="utf-8") as f:
    template = Template(f.read())
    full_prompt = template.render(
        activity=ctx.activity,
        people=ctx.people,
        time=ctx.time,
        location=ctx.location,
        video_length=getattr(ctx, "video_length", "") or "",
    )
with open("prompts/event_split_prompt.md", "r", encoding="utf-8") as f:
    template = Template(f.read())
    event_prompt = template.render(
        activity=ctx.activity,
        people=ctx.people,
        time=ctx.time,
        location=ctx.location,
        video_length=getattr(ctx, "video_length", "") or "",
    )
with open("prompts/entity_extract_prompt.md", "r", encoding="utf-8") as f:
    template = Template(f.read())
    entity_prompt = template.render(
        activity=ctx.activity,
        people=ctx.people,
        time=ctx.time,
        location=ctx.location,
        video_length=getattr(ctx, "video_length", "") or "",
    )

# 将字符串切分成token并转换为token id
tokens = tokenizer.encode(full_prompt)
print(f"经过切分后的token id为：{tokens}。")
print(f"经过切分后共有{len(tokens)}个token")

# # 将token id转化为字符串并打印出来
# for i in range(len(tokens)):
#     print(f"token id为{tokens[i]}对应的字符串为：{tokenizer.decode(tokens[i])}")

经过切分后的token id为：[2, 220, 16, 13, 5430, 9909, 88802, 27866, 56568, 44063, 54542, 46944, 99363, 17340, 24641, 87140, 88802, 3837, 106856, 60548, 87752, 77540, 99371, 101042, 48443, 99363, 100385, 5122, 115692, 99371, 198, 16, 13, 55059, 44834, 101932, 99600, 198, 17, 13, 220, 17714, 101932, 99600, 107439, 99936, 116226, 105814, 198, 18, 13, 19468, 240, 102388, 57621, 198, 19, 13, 220, 17714, 103991, 44729, 57621, 107439, 99936, 116226, 105814, 271, 99802, 100385, 5122, 101565, 99371, 9909, 100645, 18493, 103991, 44729, 57621, 101979, 60548, 23083, 20, 13, 73562, 103991, 44729, 57621, 20450, 104589, 107439, 99936, 101565, 198, 21, 13, 220, 17714, 103991, 101565, 104004, 105814, 57218, 14585, 14445, 104449, 86119, 271, 334, 100645, 100470, 59879, 2073, 60726, 115692, 99371, 87256, 101565, 99371, 97907, 107684, 75117, 1773, 334, 1406, 2, 220, 17, 13, 9608, 271, 99600, 100220, 5122, 102946, 104631, 198, 104013, 5122, 35946, 198, 20450, 5122, 17, 15, 17, 19, 7948, 21, 9754, 16, 20, 8903, 16, 